In [1]:
from geometry import GeometryMeasurement as gm

In [ ]:
import os
import sys
import warnings

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

sys.path.append(f'{os.environ["HOME"]}/pythonmodules')
sys.path.append(f'{os.environ["HOME"]}/pythonmodules/selenium')

from ics.cobraCharmer.cobraCoach import visDianosticPlot_TK as vdp
from geometry import runDir2Date


REQUIRED_COLUMNS = {
    "visit",
    "runDir",
    "arm",
    "EL",
    "rot",
    "runNumber",
    "year",
    "month",
}


def find_run_dir(visit, run_number, records, resolved):
    """Find a visit's runDir, with a same-run +/-1 visit fallback."""

    try:
        run_dir = vdp.findRunDir(int(visit))
    except Exception as exc:
        warnings.warn(
            f"Could not query runDir for visit {visit}: {exc}",
            RuntimeWarning,
        )
        run_dir = None

    if run_dir is not None:
        return str(run_dir)

    # Some forward/reverse visit pairs share one runDir but only one visit
    # can be resolved directly. Reuse a resolved neighboring visit only
    # within the same requested run number.
    neighbors = records.loc[
        records["runNumber"].eq(run_number)
        & records["visit"].sub(visit).abs().eq(1),
        "visit",
    ]
    for neighbor_visit in neighbors:
        if int(neighbor_visit) in resolved:
            return resolved[int(neighbor_visit)]

    return None


def update_visit_runDir(runVisitList, visit_runDir):
    """Update visit_runDir.csv from `runNumber visit arm EL rot` rows.

    Every visit is first resolved independently. If that fails, the function
    reuses the runDir of an already-resolved visit differing by one, provided
    both visits belong to the same run number.
    """

    raw = np.loadtxt(runVisitList, dtype=str, ndmin=2)
    if raw.shape[1] != 5:
        raise ValueError(
            f"{runVisitList} must contain five columns: "
            "runNumber, visit, arm, EL, and rot."
        )

    records = pd.DataFrame(
        raw,
        columns=["runNumber", "visit", "arm", "EL", "rot"],
    )
    records["runNumber"] = records["runNumber"].astype(int)
    records["visit"] = records["visit"].astype(int)
    records["arm"] = records["arm"].str.lower()
    records["EL"] = pd.to_numeric(records["EL"])
    records["rot"] = pd.to_numeric(records["rot"])

    invalid_arms = ~records["arm"].isin({"theta", "phi"})
    if invalid_arms.any():
        raise ValueError(
            "arm must be theta or phi; invalid rows: "
            f"{records.loc[invalid_arms].to_dict('records')}"
        )

    try:
        df = pd.read_csv(visit_runDir)
    except FileNotFoundError:
        df = pd.DataFrame(columns=sorted(REQUIRED_COLUMNS))

    for column in REQUIRED_COLUMNS:
        if column not in df.columns:
            df[column] = np.nan

    # Resolve direct matches first so +/-1 fallback is independent of row order.
    resolved = {}
    for visit in tqdm(
        records["visit"].unique(),
        desc="Resolving run directories",
    ):
        try:
            run_dir = vdp.findRunDir(int(visit))
        except Exception:
            run_dir = None
        if run_dir is not None:
            resolved[int(visit)] = str(run_dir)

    updated_rows = []
    for row in records.itertuples(index=False):
        run_dir = resolved.get(row.visit)
        if run_dir is None:
            run_dir = find_run_dir(
                row.visit,
                row.runNumber,
                records,
                resolved,
            )

        if run_dir is None:
            warnings.warn(
                f"Skipping visit {row.visit} (run {row.runNumber}): "
                "runDir could not be identified.",
                RuntimeWarning,
            )
            continue

        resolved[row.visit] = run_dir
        run_date = runDir2Date(run_dir)
        values = {
            "runNumber": row.runNumber,
            "visit": row.visit,
            "arm": row.arm,
            "EL": row.EL,
            "rot": row.rot,
            "runDir": run_dir,
            "year": run_date.year,
            "month": run_date.month,
        }

        mask = df["visit"].eq(row.visit)
        if mask.any():
            for column, value in values.items():
                df.loc[mask, column] = value
        else:
            df = pd.concat(
                [df, pd.DataFrame([values])],
                ignore_index=True,
            )
        updated_rows.append(values)

    df = df.sort_values("visit", kind="stable").reset_index(drop=True)
    df.to_csv(visit_runDir, index=False)

    print(
        f"Updated {len(updated_rows)} of {len(records)} rows in "
        f"{visit_runDir}."
    )
    return df


runVisitList = "fullVisitList2.txt"
visit_runDir = "visit_runDir2.csv"

df = update_visit_runDir(runVisitList, visit_runDir)

Resolving run directories:   0%|          | 0/117 [00:00<?, ?it/s]

In [8]:
measure = gm(
    "fullVisitList.txt",
    visitRunDirList="visit_runDir2.csv",
)

In [ ]:
measure.determineCenter()

/work/nowan/PerformanceMonitoring/geometry.py:1078: RuntimeWarning: Error while processing row 21 (18 113784 theta 90 0): No runDir could be found for visit 113784.; skipping it.
  warnings.warn(
/work/nowan/PerformanceMonitoring/geometry.py:1078: RuntimeWarning: Error while processing row 65 (23 127886 theta 60 0): No runDir could be found for visit 127886.; skipping it.
  warnings.warn(
/work/nowan/PerformanceMonitoring/geometry.py:1078: RuntimeWarning: Error while processing row 66 (23 127888 phi 60 0): No runDir could be found for visit 127888.; skipping it.
  warnings.warn(
/work/nowan/PerformanceMonitoring/geometry.py:1078: RuntimeWarning: Error while processing row 67 (23 127882 theta 30 0): No runDir could be found for visit 127882.; skipping it.
  warnings.warn(
/work/nowan/PerformanceMonitoring/geometry.py:1078: RuntimeWarning: Error while processing row 174 (30 150010 theta 90 120): No runDir could be found for visit 150010.; skipping it.
  warnings.warn(


Performing fits for 20240503_012:   0%|          | 0/2394 [00:00<?, ?it/s]

In [20]:
measure.determineLength()

/work/nowan/PerformanceMonitoring/geometry.py:673: RuntimeWarning: Skipping run22/EL_30/rot_0: both arms are required.
  warnings.warn(
/work/nowan/PerformanceMonitoring/geometry.py:673: RuntimeWarning: Skipping run27/EL_30/rot_0: both arms are required.
  warnings.warn(
/work/nowan/PerformanceMonitoring/geometry.py:673: RuntimeWarning: Skipping run29/EL_30/rot_0: both arms are required.
  warnings.warn(


PosixPath('geometry/Geometry.hdf5')

In [21]:
measure.determineMovementLimits()

/work/nowan/PerformanceMonitoring/geometry.py:857: RuntimeWarning: Skipping geometry/thetaGeometry.hdf5/20230421_000: missing attributes ['rot', 'runNumber'].
  warnings.warn(
/work/nowan/PerformanceMonitoring/geometry.py:1078: RuntimeWarning: Error while processing geometry/thetaGeometry.hdf5/20250528_009 while calculating movement limits: "merged geometry is missing ['cobraIdx', 'opening_angle', 'phi']"; skipping it.
  warnings.warn(
/work/nowan/PerformanceMonitoring/geometry.py:1078: RuntimeWarning: Error while processing geometry/thetaGeometry.hdf5/20260324_011 while calculating movement limits: "merged geometry is missing ['cobraIdx', 'opening_angle', 'phi']"; skipping it.
  warnings.warn(
/work/nowan/PerformanceMonitoring/geometry.py:1078: RuntimeWarning: Error while processing geometry/thetaGeometry.hdf5/20260324_013 while calculating movement limits: "merged geometry is missing ['cobraIdx', 'opening_angle', 'phi']"; skipping it.
  warnings.warn(
/work/nowan/PerformanceMonitorin

{'theta': ['20240503_012',
  '20240506_015',
  '20240507_012',
  '20240507_022',
  '20240529_032',
  '20240530_011',
  '20240531_012',
  '20240830_005',
  '20240830_011',
  '20240830_020',
  '20241025_007',
  '20241028_003',
  '20241028_023',
  '20250124_011',
  '20250127_008',
  '20250127_012',
  '20250321_005',
  '20250321_011',
  '20250321_017',
  '20250519_005',
  '20250521_015',
  '20250521_019',
  '20250523_011',
  '20250526_014',
  '20250620_006',
  '20250926_034',
  '20250926_037',
  '20250926_039',
  '20260116_001',
  '20260116_002',
  '20260116_014',
  '20260126_015',
  '20260126_016',
  '20260126_017',
  '20260127_001',
  '20260127_002',
  '20260127_003',
  '20260312_002',
  '20260312_005',
  '20260312_008',
  '20260325_001',
  '20260325_003',
  '20260325_007',
  '20260711_012',
  '20260713_013',
  '20260714_013',
  '20260716_015',
  '20260718_025',
  '20260718_028',
  '20260906_013',
  '20260906_015',
  '20260906_017',
  '20260910_003',
  '20260910_005',
  '20260910_007',
 

In [ ]:
from mm import MotorMapMeasurement as mm

measure = mm(
    visit='run29.txt',
    arm="theta",
    EL=90,
)

In [ ]:
measure.determineMotorMap()

In [ ]:
measure.combineMotorMaps()

In [4]:
import glob
glob.glob("geometry/*")

['geometry/thetaGeometry.hdf5',
 'geometry/phiGeometry.hdf5',
 'geometry/Geometry.hdf5']

In [19]:
import h5py
with h5py.File("geometry/phiGeometry.hdf5", 'r') as f:
    print(f['20260912_032'].attrs["runNumber"])

30


In [5]:
1

1